### Layout test using PP-DocLayout_plus-L

In [1]:

#gpu memory not suffiicient 
import os
os.environ["FLAGS_fraction_of_gpu_memory_to_use"] = "0.6"
os.environ["FLAGS_eager_delete_tensor_gb"] = "0.0"



In [2]:
from paddleocr import PaddleOCR, LayoutDetection, FormulaRecognition

layout_model = LayoutDetection(
    model_name="PP-DocLayout_plus-L",
    threshold={2: 0.2, 12: 0.2, 14: 0.2},  # text, header, footer
    device='gpu:0',
    layout_nms=True,
    layout_merge_bboxes_mode= "large",
    layout_unclip_ratio =1.1
)
formula_model = FormulaRecognition()
ocr_model = PaddleOCR(
    text_detection_model_name="PP-OCRv5_server_det",
    text_recognition_model_name="PP-OCRv5_server_rec",
    lang='en',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True,
    device="gpu:0"
    
)

Using official model (PP-DocLayout_plus-L), the model files will be automatically downloaded and saved in `C:\Users\Legion\.paddlex\official_models\PP-DocLayout_plus-L`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

inference.yml: 0.00B [00:00, ?B/s]

inference.json: 0.00B [00:00, ?B/s]

... (14 outputs left out)

inference.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

inference.pdiparams:   0%|          | 0.00/84.4M [00:00<?, ?B/s]

In [3]:
import os
from pathlib import Path
import cv2
import time
import json

# def polygon_to_bbox(polygon):
#     # converts polygon to bbox coordinates
#     x_coords = [point[0] for point in polygon]
#     y_coords = [point[1] for point in polygon]
#     return [min(x_coords), min(y_coords), max(x_coords), max(y_coords)]

# def boxes_overlap(box1, box2, threshold=0.5):
 
#     # Convert to [x1, y1, x2, y2] format 
#     x1_1, y1_1, x2_1, y2_1 = box1[0], box1[1], box1[2], box1[3]
#     x1_2, y1_2, x2_2, y2_2 = box2[0], box2[1], box2[2], box2[3]
    
#     # intersection calculation
#     x1_i = max(x1_1, x1_2)
#     y1_i = max(y1_1, y1_2)
#     x2_i = min(x2_1, x2_2)
#     y2_i = min(y2_1, y2_2)
    
#     if x2_i < x1_i or y2_i < y1_i:
#         return False
    
#     # intersection calculateion
#     intersection_area = (x2_i - x1_i) * (y2_i - y1_i)
#     box1_area = (x2_1 - x1_1) * (y2_1 - y1_1)
    
#     # Check if box1 is mostly inside box2
#     overlap_ratio = intersection_area / box1_area if box1_area > 0 else 0
    
#     return overlap_ratio > threshold


In [4]:
def run_image_pipeline_nc(path, layout_model, ocr):
    image_path = Path(path)
    now = time.time()
 
    # --- Layout analysis ---
    output_layout = layout_model.predict(str(image_path))
    time_after_layout = time.time()
    
    # --- OCR ---
    output_ocr = ocr.predict(str(image_path))
    time_after_ocr = time.time()
    
    # --- output directories 
    layout_dir = Path("output/pipeline_test_nc/layout")
    ocr_dir = Path("output/pipeline_test_nc/ocr")
 
    layout_dir.mkdir(parents=True, exist_ok=True)
    ocr_dir.mkdir(parents=True, exist_ok=True)   
  
    # --- filtering OCR results ---
    for res in output_ocr:
        # Save original OCR
        img_file = ocr_dir / f"{image_path.stem}_ocr.png"
        json_file = ocr_dir / f"{image_path.stem}_ocr.json"
        res.save_to_img(str(img_file))
        res.save_to_json(str(json_file))
            
    # --- Save Layout results ---
    for res in output_layout:
        img_file = layout_dir / f"{image_path.stem}_layout.png"
        json_file = layout_dir / f"{image_path.stem}_layout.json"
        res.save_to_img(str(img_file))
        res.save_to_json(str(json_file))
     
    print(f"Total processed in {time_after_ocr - now:.2f} seconds")
    print(f"Layout analysis: {time_after_layout - now:.2f} s")
    print(f"OCR: {time_after_ocr - time_after_layout:.2f} s")
 

In [9]:
run_image_pipeline_nc("../data/table1.jpg", layout_model, ocr_model)

The result has multiple img files need to be saved. But the `save_path` has been specified as `output\pipeline_test_nc\ocr\table1_ocr.png`!


Total processed in 4.13 seconds
Layout analysis: 2.32 s
OCR: 1.81 s


In [ ]:
run_image_pipeline_nc("data/sample_image4.png", layout_model, ocr_model)

The result has multiple img files need to be saved. But the `save_path` has been specified as `output\pipeline_test_nc\ocr\sample_image4_ocr.png`!


Total processed in 1.19 seconds
Layout analysis: 0.07 s
OCR: 1.12 s


### pp structurev3 end to end try out

In [ ]:
#!pip install "paddlex[ocr]==3.2.1"
#version should be the same as pip show paddlex

In [ ]:
from paddlex import create_pipeline
from pathlib import Path
import time 

output_path = Path("output/structureV3")
pipeline = create_pipeline(pipeline="PP-StructureV3")



Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\UVDoc`.
Creating model: ('PP-DocBlockLayout', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-DocBlockLayout`.
Creating model: ('PP-DocLayout_plus-L', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-DocLayout_plus-L`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-LCNet

In [ ]:
now = time.time()
output = pipeline.predict(
    input="data/sample_pdf2.pdf",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)

output_path.mkdir(parents = True, exist_ok = True)
for res in output:
    res.print() ## Print the structured prediction output
    res.save_to_json(save_path=output_path) ## Save the structured JSON result of the current image
    res.save_to_markdown(save_path=output_path) ## Save the result of the current image in Markdown format
print(f"time taken: {(time.time() - now):.2f}")

Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-OCRv5_server_rec`.
{'res': {'input_path': 'data/sample_pdf2.pdf', 'page_index': 0, 'model_settings': {'use_doc_preprocessor': False, 'use_seal_recognition': False, 'use_tabl

time taken: 41.89


### Layout Parsing model 

In [ ]:
from paddlex import create_pipeline

pipeline_par = create_pipeline(pipeline="layout_parsing")

Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\UVDoc`.
Creating model: ('RT-DETR-H_layout_17cls', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\RT-DETR-H_layout_17cls`.
Creating model: ('PP-OCRv4_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-OCRv4_server_det`.
Creating model: ('PP-OCRv4_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-OC

In [ ]:
output_path_parsing = Path("output/layout_parsing")
output_path_parsing.mkdir(parents = True, exist_ok = True)
now = time.time()    
output_par = pipeline_par.predict(
    input="data/page_1.png",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)
for res in output_par:
    res.print()  ## Print the structured output of the prediction
    res.save_to_img(save_path="output/layout_parsing")  ## Save the visualized image results of all submodules for the current image
    res.save_to_json(save_path="output/layout_parsing")  ## Save the structured JSON results for the current image
    res.save_to_xlsx(save_path="output/layout_parsing")  ## Save the sub-table results in XLSX format for the current image
    res.save_to_html(save_path="output/layout_parsing")  ## Save the sub-table results in HTML format for the current image
print(f"time taken: {(time.time() - now):.2f}")

{'res': {'input_path': 'data/page_1.png', 'page_index': None, 'model_settings': {'use_doc_preprocessor': False, 'use_seal_recognition': True, 'use_table_recognition': True, 'use_formula_recognition': False}, 'parsing_res_list': [{'block_bbox': [np.float32(176.91135), np.float32(108.531944), np.float32(1821.0555), np.float32(293.98557)], 'block_label': 'image', 'block_content': 'eurostat\n****+\neuroindicators'}, {'block_bbox': [np.float32(1676.9105), np.float32(473.00583), np.float32(2322.2659), np.float32(527.1471)], 'block_label': 'text', 'block_content': '51/2022 － 29 April2022'}, {'block_bbox': [np.float32(166.7516), np.float32(690.2386), np.float32(1637.8815), np.float32(848.78375)], 'block_label': 'text', 'block_content': 'Flash estimate - April 2022\nEuro area annual inflation up to 7.5%'}, {'block_bbox': [np.float32(166.7516), np.float32(690.2386), np.float32(1637.8815), np.float32(848.78375)], 'block_label': 'doc_title', 'block_content': 'Flash estimate - April 2022\nEuro area

time taken: 9.45


In [ ]:
import cv2
import numpy as np

def preprocess_region(img, x1, y1, x2, y2, pad_w=5, pad_h=5, min_height=32):
      # Add padding with boundary checks
    x1_pad = max(0, x1 - pad_w)
    y1_pad = max(0, y1 - pad_h)
    x2_pad = min(img.shape[1], x2 + pad_w)
    y2_pad = min(img.shape[0], y2 + pad_h)
    
    # Crop region
    region = img[y1_pad:y2_pad, x1_pad:x2_pad]
    
    # Upscale if region is too small
    h, w = region.shape[:2]
    if h < min_height:
        scale = min_height / h
        new_width = int(w * scale)
        region = cv2.resize(region, (new_width, min_height), interpolation=cv2.INTER_LINEAR)
    
    return region

In [ ]:
import os
import json
import time
import cv2
import numpy as np
from pathlib import Path
from paddleocr import FormulaRecognition

def run_image_pipeline_c(path, preprocessing_pipeline, layout_model, formula_model, ocr_model):
    image_path = Path(path)
    start_time = time.time()
    image_name = image_path.stem

    # --- Step 1: Layout detection ---
    layout_results = layout_model.predict(str(image_path))
    layout_time = time.time()

    # Create directories
    layout_dir = Path(f"output/pipeline_test/{image_name}/layout")
    ocr_dir = Path(f"output/pipeline_test/{image_name}/ocr")
    formula_dir = Path(f"output/pipeline_test/{image_name}/formula")
    for d in [layout_dir, ocr_dir, formula_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # layout model and its output
    for res in layout_results:
        layout_img = layout_dir / f"{image_name}_layout.png"
        layout_json = layout_dir / f"{image_name}_layout.json"
        res.save_to_img(str(layout_img))
        res.save_to_json(str(layout_json))

    # 
    img = cv2.imread(str(image_path))

    # Prepare OCR JSON output
    ocr_json_path = ocr_dir / f"{image_name}_ocr.json"
    if ocr_json_path.exists():
        with open(ocr_json_path, "r", encoding="utf-8") as f:
            all_ocr_results = json.load(f)
    else:
        all_ocr_results = {}

    SKIP_FOR_OCR = ["table", "figure_table_title", "chart", "image", "seal", "formula"]

    #loop through layout boxes
    for page_result in layout_results:
        for i, box in enumerate(page_result["boxes"]):
            cls = box['label']
            conf = box['score']
            coords = [int(x) for x in box['coordinate']]
            x1, y1, x2, y2 = coords
            
            region = preprocess_region(img, x1, y1, x2, y2)
            # --- OCR Regions preprocessed and cropped 
            if (cls not in SKIP_FOR_OCR):
                ocr_result = ocr_model.predict(region)
                out_img = ocr_dir / f"{image_name}_{cls}_{i}.png"
                cv2.imwrite(str(out_img), region)

                # Store OCR results per region
                if f"region_{i}" not in all_ocr_results:
                    all_ocr_results[f"region_{i}"] = []

                recognized_polys =[]
                for res in ocr_result:
                    structured = {}
                    structured["class"] = cls
                    structured["confidence"] = float(conf)
                    structured["bbox"] = [x1, y1, x2, y2]  # original region box
                    structured["recognized_texts"] = res['rec_texts']
                    structured["recognized_confidence"] = [float(score) for score in res['rec_scores']]
                    
                    for poly in res["rec_polys"]:
                         abs_poly = [[p[0] + x1, p[1] + y1] for p in poly]
                         recognized_polys.append(abs_poly)
                    structured["recognized_polys"] = [
                              poly.tolist() if isinstance(poly, np.ndarray) else poly
                                for poly in res['rec_polys']
                      ] 
                    all_ocr_results[f"region_{i}"]=structured

            # --- formula Regions evaluated by special model ---
            elif cls == "formula":
                formula_output = formula_model.predict(region)
                for idx, f_res in enumerate(formula_output):
                    formula_img_path = formula_dir / f"{image_name}_formula_{i}_{idx}.png"
                    formula_json_path = formula_dir / f"{image_name}_formula_{i}_{idx}.json"
                    f_res.save_to_img(save_path=str(formula_img_path))
                    f_res.save_to_json(save_path=str(formula_json_path))

    # Save OCR JSON
    with open(ocr_json_path, "w", encoding="utf-8") as f:
        json.dump(all_ocr_results, f, ensure_ascii=False, indent=4)

    end_time = time.time()
    print(f"Total Time: {end_time - start_time:.2f}s")
    print(f"Layout Detection Time: {layout_time - start_time:.2f}s")
    print(f"OCR + Formula Processing Time: {end_time - layout_time:.2f}s")


In [ ]:
run_image_pipeline_c(
    path="data/sample_image4.png",
    preprocessing_pipeline=None, 
    layout_model=layout_model,
    formula_model=formula_model,
    ocr_model=ocr_model
)

Total Time: 3.35s
Layout Detection Time: 0.16s
OCR + Formula Processing Time: 3.19s


In [ ]:
run_image_pipeline_c(
    path="data/page_1.png",
    preprocessing_pipeline=None,  
    layout_model=layout_model,
    formula_model=formula_model,
    ocr_model=ocr_model
)

Total Time: 1.80s
Layout Detection Time: 0.15s
OCR + Formula Processing Time: 1.66s
